### ETL: bronze.air_quality_history -> silver.air_quality_history_cleaned

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, explode, expr
from pyspark.sql.types import  FloatType, DateType, StringType
from delta.tables import DeltaTable 
import sys
import os
from pathlib import Path
current_dir = "/Workspace" + os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
src_path = str(Path(current_dir).parents[1])
sys.path.append(src_path)

from utils.cleaning_functions import *

In [0]:
df_source = spark.table("dbw_routemind_euskadi_dev.bronze.air_quality_history")

In [0]:
notnull_columns = [
    "date",
    "sensorId"
]


In [0]:
df_exploded = df_source.select(
    col("date"),
    explode(col("station")).alias("station_element")
)

In [0]:
df_extracted = df_exploded.select(
    col("date").cast(T.DateType()).alias("date"),
    col("station_element.id").cast(T.StringType()).alias("sensorId"),
    expr("filter(station_element.measurements, x -> x.name = 'PM10')[0].value").cast(T.FloatType()).alias("measure_PM10"),
    expr("filter(station_element.measurements, x -> x.name = 'PM2,5')[0].value").cast(T.FloatType()).alias("measure_PM2_5")
)

In [0]:
df_filtered = df_extracted.filter(
    col("measure_PM10").isNotNull() | col("measure_PM2_5").isNotNull()
)

In [0]:
df_clean = drop_null_required(df_filtered, notnull_columns)

In [0]:
df_clean.printSchema()

In [0]:
target_table = "dbw_routemind_euskadi_dev.silver.air_quality_history_cleaned"
delta_path = "abfss://silver@stroutemindeuskadidev.dfs.core.windows.net/delta_tables/air_quality_history/data"

df_clean.write \
    .format("delta") \
    .option("path", delta_path) \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable(target_table)


print(f"APPEND completed on {target_table}. rows processed: {df_clean.count()}")

In [0]:
%sql
select count(*) from dbw_routemind_euskadi_dev.silver.air_quality_history_cleaned